
# EDA on English Premier League Players Game Statistics 
---

## Objectives:

- Exploratory data anlysis on English Premier League players dataset.
- Learn and apply data visualizatiosn techniques using plotly's data visualization library.

---
<a id="top"></a>

## Table of Contents
* [1. Introduction](#1)
    * [1.1 The Premier League](#1.1)
    * [1.2 The Dataset](#1.2)
    * [1.3 Data Pre-Processing](#1.3)
* [2. General Statistics](#2)
    * [2.1 Countries most represented in the EP](#2.1)
    * [2.2 Players Appearances (nr. games](#2.2)
    * [2.3 Players' Age](#2.3)
* [3. Players Stats By Playing Position](#3)
    * [3.1 Goalkeepers](#3.1)
    * [3.2 Defenders](#3.2)
    * [3.3 Midfielders](#3.3)
    * [3.4 Forwards](#3.4)
* [4. Other Statistics](#4)
    * [4.1 Goal Distribution](#4.1)
    * [4.2 Unwanted records](#4.2)
* [5. Closing Remarks](#5)   
* [6. References](#5)  
---


## 1. Introduction <a class="anchor" id="1"></a>
### 1.1 The Premier League <a class="anchor" id="1.1"></a>
The Premier League, often referred outside England as the `English Premier League or the EPL for short`, is the top level of the English football league system. Contested by 20 clubs, it operates on a system of promotion and relegation with the English Football League (EFL). Seasons run from August to May with each team playing 38 matches (playing all 19 other teams both home and away). [[source](https://en.wikipedia.org/wiki/Premier_League)]

### 1.2 The Dataset <a class="anchor" id="1.2"></a>

The [dataset](https://www.kaggle.com/rishikeshkanabar/premier-league-player-statistics-updated-daily) is correct upto and including 2020-09-24 (a lot has has happened since then, notebook will be updated when the dataset gets an update). Each row in the data represents a football player currently playing in the EPL and the columns are features used to discribe players' data and game statistics. There are 571 rows and 59 columns in the data. Few of the columns are:

* `Name`: Name of the player
* `Club`: Club the player plays for at present
* `Position`: Playing position(Goalkeeper, Defender, Mid-fider, Forward)
* `Nationality`: Country the player is from
* `Age`: Players age
* `Appearances`: Number of games played (a substitute appearance aslo counts)
* `Wins`: Number of games the palyer has won
* `Losses`: Number of games the palyer has lost
* `Goals`: Number of goals the player has scored in the EPL 
* `Goals` per match: Goals scored per game
* `etcetra`.


**Notes** : 

1. A player's attributes from previous premier league clubs are carried to his current club. 
2. When all-time stats are considered, obviously longevity in the game plays a big role. The longer the player has played in the EPL, the higher the stats (count of something for example) are going to be. Hence the *per-game* stat will be a better indicator of performance. On the other hand players who played too few games might appear as top-performers in thier per-game statistics. For this reason only players who played at least 38 games are considered (a full-season's worth of games) in the per-game comparison.


In [41]:
from itertools import repeat
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd 
pd.set_option('mode.chained_assignment', None)
import seaborn as sns
import plotly.io as pio
pio.renderers.default = 'iframe'
import plotly.express as px
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected=True)

data = pd.read_csv('data/dataset - 2020-09-24.csv')
data = data.copy()

print('Shape of the dataset is {}'.format(data.shape))
data.head()

Shape of the dataset is (571, 59)


,Name,Jersey Number,Club,Position,Nationality,Age,Appearances,Wins,Losses,Goals,...,Punches,High Claims,Catches,Sweeper clearances,Throw outs,Goal Kicks,Yellow cards,Red cards,Fouls,Offsides
0,Bernd Leno,1.0,Arsenal,Goalkeeper,Germany,28.0,64,28,16,0,...,34.0,26.0,17.0,28.0,375.0,489.0,2,0,0,NaN
1,Matt Macey,33.0,Arsenal,Goalkeeper,England,26.0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,NaN
2,Rúnar Alex Rúnarsson,13.0,Arsenal,Goalkeeper,Iceland,25.0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,NaN
3,Héctor Bellerín,2.0,Arsenal,Defender,Spain,25.0,160,90,37,7,...,NaN,NaN,NaN,NaN,NaN,NaN,23,0,125,8.0
4,Kieran Tierney,3.0,Arsenal,Defender,Scotland,23.0,16,7,5,1,...,NaN,NaN,NaN,NaN,NaN,NaN,2,0,9,0.0


In [42]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 571 entries, 0 to 570
Data columns (total 59 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Name                    571 non-null    str    
 1   Jersey Number           563 non-null    float64
 2   Club                    571 non-null    str    
 3   Position                571 non-null    str    
 4   Nationality             570 non-null    str    
 5   Age                     570 non-null    float64
 6   Appearances             571 non-null    int64  
 7   Wins                    571 non-null    int64  
 8   Losses                  571 non-null    int64  
 9   Goals                   571 non-null    int64  
 10  Goals per match         309 non-null    float64
 11  Headed goals            502 non-null    float64
 12  Goals with right foot   502 non-null    float64
 13  Goals with left foot    502 non-null    float64
 14  Penalties scored        309 non-null    float64
 15  

### 1.3 Data pre-processing <a class="anchor" id="1.3"></a>

In [43]:
# Remove entries without nationality, age, or jersey number
data = data.dropna(subset=['Nationality', 'Age', 'Jersey Number'])

# =========================
# CLEAN PERCENTAGE COLUMNS
# =========================

percent_cols = [
    'Cross accuracy %',
    'Shooting accuracy %',
    'Tackle success %'
]

for col in percent_cols:

    # Remove % sign only if column is object/string
    if data[col].dtype == 'object':
        data[col] = data[col].str.replace('%', '', regex=False)

    # Convert to float
    data[col] = pd.to_numeric(data[col], errors='coerce')

# =========================
# CREATE CLEAN DATAFRAME
# =========================

features = data.columns
data_clean = data[features]

# Avoid division by zero
data_clean_appNonZero = data_clean[
    data_clean['Appearances'] > 0
].copy()

# =========================
# NORMALIZE STATISTICS
# =========================

# Columns that should NOT be divided
exclude_cols = [
    'Age',
    'Name',
    'Appearances',
    'Club',
    'Nationality',
    'Jersey Number',
    'Cross accuracy %',
    'Position',
    'Goals per match',
    'Passes per match',
    'Tackle success %',
    'Shooting accuracy %'
]

# Columns to divide by appearances
cols = features.drop(exclude_cols)

# Convert to float before division
data_clean_appNonZero[cols] = (
    data_clean_appNonZero[cols]
    .astype(float)
    .div(data_clean_appNonZero['Appearances'], axis=0)
)

# =========================
# POSITIONAL CLASSIFICATION
# =========================

goalies = data[data['Position'] == 'Goalkeeper']
defenders = data[data['Position'] == 'Defender']
midfielders = data[data['Position'] == 'Midfielder']
forwards = data[data['Position'] == 'Forward']

# =========================
# PLAYERS WITH >= 38 APPEARANCES
# =========================

# Original data
data_38app = data[data['Appearances'] >= 38]

goalies_38app = goalies[goalies['Appearances'] >= 38]
defenders_38app = defenders[defenders['Appearances'] >= 38]
midfielders_38app = midfielders[midfielders['Appearances'] >= 38]
forwards_38app = forwards[forwards['Appearances'] >= 38]

# =========================
# NORMALIZED DATA (>=38 APPEARANCES)
# =========================

all_players = data_clean_appNonZero[
    data_clean_appNonZero['Appearances'] >= 38
]

goalies_ = data_clean_appNonZero[
    (data_clean_appNonZero['Position'] == 'Goalkeeper') &
    (data_clean_appNonZero['Appearances'] >= 38)
]

defenders_ = data_clean_appNonZero[
    (data_clean_appNonZero['Position'] == 'Defender') &
    (data_clean_appNonZero['Appearances'] >= 38)
]

midfielders_ = data_clean_appNonZero[
    (data_clean_appNonZero['Position'] == 'Midfielder') &
    (data_clean_appNonZero['Appearances'] >= 38)
]

forwards_ = data_clean_appNonZero[
    (data_clean_appNonZero['Position'] == 'Forward') &
    (data_clean_appNonZero['Appearances'] >= 38)
]

# =========================
# PREVIEW
# =========================

print(data_clean_appNonZero.head())

              Name  Jersey Number     Club    Position Nationality   Age  \
0       Bernd Leno            1.0  Arsenal  Goalkeeper     Germany  28.0   
3  Héctor Bellerín            2.0  Arsenal    Defender       Spain  25.0   
4   Kieran Tierney            3.0  Arsenal    Defender    Scotland  23.0   
6         Sokratis            5.0  Arsenal    Defender      Greece  32.0   
7      Rob Holding           16.0  Arsenal    Defender     England  25.0   

   Appearances      Wins    Losses     Goals  ...  Punches  High Claims  \
0           64  0.437500  0.250000  0.000000  ...  0.53125      0.40625   
3          160  0.562500  0.231250  0.043750  ...      NaN          NaN   
4           16  0.437500  0.312500  0.062500  ...      NaN          NaN   
6           44  0.477273  0.250000  0.068182  ...      NaN          NaN   
7           41  0.609756  0.219512  0.000000  ...      NaN          NaN   

    Catches  Sweeper clearances  Throw outs  Goal Kicks  Yellow cards  \
0  0.265625        

<a href="#top">Back to top</a>  

## 2. General Statistics <a class="anchor" id="2"></a>
### 2.1 Countries most represented in the EPL <a class="anchor" id="2.1"></a>

In any league it is normal to have more home-grown players than foreign palyers and the EPL is no different. The majority of the players are English. Other UK member countries will also likely have more representations. The question is which country comes next? The first thing that can be a factor is geographyical proximity. What comes after that would be likely to be determined by talent baring workpermit issue and visa-related challenges that could prevent some players from playing/working in the EPL. But that is a rarity. Below are the top three nations after home country (England) ranked by overall apprearances and further breakdown by players playing positions.

#### Summary most represented nations:

* Overall: 1st **France**, 2nd **Spain**, 3rd *Brazil*
* Goalkeepers: 1st **Spain**, 2nd Denmark, 3rd **France**
* Defenders: 1st **Spain**, 2nd Nederland, 3rd *Ireland*
* Midfielders: 1st *Scotland*, 2nd **France**, 3rd **Spain**
* Forwards: 1st *Brazil*, 2nd **France**, 3rd *Ireland*

**French** and **Spanish** players found their second home in England. Honorable mention to **Brazil**

In [44]:
import plotly.express as px

df = data

fig = px.pie(
    df,
    values='Appearances',
    names='Nationality',
    title='Countries represented in the EPL by number of appearances'
)

# text inside pie chart
fig.update_traces(
    textposition='inside',
    textinfo='percent+label'
)

# layout customization
fig.update_layout(
    title={
        'text': '<b>Nr. of appearances per country</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {
            'color': 'black',
            'size': 28,
            'family': 'Courier New'
        }
    },
    width=700,
    height=700,
    showlegend=False
)

fig.show()

In [45]:
import plotly.express as px

df = data

fig = px.sunburst(
    df,
    path=['Position', 'Nationality'],
    values='Appearances'
)

# update layout
fig.update_layout(
    title={
        'text': '<b>Players Position by Country</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {
            'color': 'black',
            'size': 28,
            'family': 'Courier New'
        }
    },
    width=700,
    height=700,
    showlegend=False
)

fig.show()

### 2.2 Players Appearances (nr. games) <a class="anchor" id="2.2"></a>

Longevity, versatility, quality of the player & the squad and  playing position are major factors for number of appearances.  

Most of the appearances come from defence and midfield position. No surprise here. If the most common line-ups/systems (4-4-2, 4-5-1, 3-5-2, 4-3-3) are averaged they would look like in the graph below. Usually midfielders are versatile and can play in defence or in attack if needed. So they constitute the majority of a given squad.

#### Summary most appearances

- Goalkeeper: Joe Hart, 340
- Defender: Phil Jagielka, 366
- Midfielder: James Milner, 539
- Forward: Theo Walcott, 346

In [46]:
import plotly.express as px

df = data

fig = px.bar(
    df,
    x="Position",
    y="Appearances",
    color='Club',
    hover_data=["Name"],
    width=750,
    height=600
)

# update layout
fig.update_layout(
    template='ggplot2',
    title={
        'text': '<b>Players Appearance by Position</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {
            'size': 24
        }
    }
)

fig.show()

In [47]:
import plotly.express as px

# =========================
# GROUP DATA BY CLUB & POSITION
# =========================

club_app = (
    df.groupby(['Club', 'Position'])['Appearances']
    .sum()
    .reset_index()
)

# =========================
# CREATE BAR CHART
# =========================

fig = px.bar(
    club_app,
    y='Club',
    x='Appearances',
    color='Position',
    orientation='h',
    width=900,
    height=700,
    hover_data=['Position'],
    template='ggplot2'
)

# =========================
# CUSTOMIZE LAYOUT
# =========================

fig.update_layout(
    title={
        'text': '<b>Appearances by Club</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {
            'size': 26,
            'family': 'Courier New',
            'color': 'black'
        }
    },

    xaxis_title='Total Appearances',
    yaxis_title='Club',

    legend_title='Position',

    font=dict(size=13)
)

# =========================
# SHOW FIGURE
# =========================

fig.show()

### 2.3 Players' Age <a class="anchor" id="2.3"></a>
Goalkeepers and defender are relatively older than thier mid-field and attacking colleagues. The defence line (including goalkeepers) is widely regarded as the area where more of a wise-head/cool-headed than a lightning fast leg is need. No harm if a defender is fast, there are many of them. But most defenders mature with age. Being an older defender is not a bad thing.

**Summary:**
* The yougest squad is Leeds-United
* Liverpool are the oldest group of players
* The min, median and max ages are 17, 25.8 and 38 years

**Extra info:**

The youngest ever EPL player: Harvey Elliott – 16 years and 30 days (Liverpool, made debut on May 2019)

The olderst ever EPL player: John Burridge – 43 years and 163 days (Aston Villa, played last 1995)

<a href="#top">Back to top</a>  

In [48]:
import plotly.express as px

df = data

# =========================
# AVERAGE AGE
# =========================

age_avg = df['Age'].mean()

# =========================
# CREATE VIOLIN PLOT
# =========================

fig1 = px.violin(
    df,
    y="Age",
    x="Position",
    box=True,
    color="Position",
    width=800,
    height=500,
    template='simple_white',
    title='<b>Players Age Distribution by Position</b>'
)

# =========================
# ADD AVERAGE AGE LINE
# =========================

fig1.add_shape(
    type="line",
    line_color="blue",
    line_width=3,
    opacity=1,
    line_dash="dot",

    x0=0,
    x1=1,
    xref="paper",

    y0=age_avg,
    y1=age_avg,
    yref="y"
)

# =========================
# ADD ANNOTATION
# =========================

fig1.add_annotation(
    x=1,
    y=age_avg,
    xref="paper",
    yref="y",

    text=f"Average Age = {age_avg:.1f}",
    showarrow=False,

    font=dict(
        size=12,
        color="blue"
    )
)

# =========================
# UPDATE LAYOUT
# =========================

fig1.update_layout(
    title={
        'x': 0.5,
        'xanchor': 'center',
        'font': {
            'size': 24
        }
    },

    xaxis_title='Position',
    yaxis_title='Age'
)

# =========================
# SHOW FIGURE
# =========================

fig1.show()


In [49]:
import plotly.express as px

# =========================
# CALCULATE AVERAGE AGE
# =========================

age_avg = df['Age'].mean()

# =========================
# CREATE BOX PLOT
# =========================

fig2 = px.box(
    df,
    y="Club",
    x="Age",
    color="Club",
    width=900,
    height=800,
    template='ggplot2',
    title='<b>Players Age Distribution by Club</b>'
)

# =========================
# ADD AVERAGE AGE LINE
# =========================

fig2.add_shape(
    type="line",

    line_color="black",
    line_width=3,
    opacity=1,
    line_dash="dot",

    y0=0,
    y1=1,
    yref="paper",

    x0=age_avg,
    x1=age_avg,
    xref="x"
)

# =========================
# ADD ANNOTATION
# =========================

fig2.add_annotation(
    x=age_avg,
    y=1,
    yref="paper",

    text=f"Average Age = {age_avg:.1f}",
    showarrow=False,

    font=dict(
        size=12,
        color="black"
    )
)

# =========================
# UPDATE LAYOUT
# =========================

fig2.update_layout(

    title={
        'x': 0.5,
        'xanchor': 'center',
        'font': {
            'size': 24
        }
    },

    xaxis_title='Age',
    yaxis_title='Club',

    showlegend=False
)

# =========================
# OPTIONAL: SORT CLUBS
# =========================

# fig2.update_layout(
#     yaxis={'categoryorder':'total descending'}
# )

# =========================
# SHOW FIGURE
# =========================

fig2.show()

## 3. Players Stats By Playing Position <a class="anchor" id="3"></a>
### 3.1 Goalkeepers  <a class="anchor" id="3.1"></a>

One of the key stats for goalkeepers is the much coveted `clean sheet` (conceeding zero goals in a game). Although this stat is not entierly dependent on the performance/ability of the goalkeeper only (a solid defence-line infront of a goalkeeper always helps), this metric shows how good a goal keeper is. The goalkeeper who kept the most clean sheets is awarded a golden-glove for thier effort at the end of a season. Other important qualities of a goalkeeper are:

*  Clean sheets
*  Saves
*  Penalties saved 
*  Punches 
*  High Claims 
*  Catches

**Extra info**: 
Most Expensive goal keepers in the EPL are:
1. Kepa Arrizabalaga (SPN), Athletic Bilbao to Chelsea in 2018, £71m
2. Alisson Becker (BRA), AS Roma to Liverpool in 2018, £65m
3. Ederson Moraes (BRA), Benfica to Manchester City in 2017, £34.7m

[Here is the reference. ](https://www.espn.com/soccer/soccer-transfers/story/3135816/the-10-most-expensive-goalkeepers-kepa-alisson-becker-courtois-ederson)


In [50]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# =========================
# TOP N PLAYERS
# =========================

head = 5

# -------------------------
# OVERALL STATS
# -------------------------

df1 = goalies_38app.sort_values(
    by='Clean sheets',
    ascending=False
).head(head)

df2 = goalies_38app.sort_values(
    by='Saves',
    ascending=False
).head(head)

df3 = goalies_38app.sort_values(
    by='High Claims',
    ascending=False
).head(head)

df4 = goalies_38app.sort_values(
    by='Catches',
    ascending=False
).head(head)

# -------------------------
# PER GAME STATS
# -------------------------

df11 = goalies_.sort_values(
    by='Clean sheets',
    ascending=False
).head(head)

df12 = goalies_.sort_values(
    by='Saves',
    ascending=False
).head(head)

df13 = goalies_.sort_values(
    by='High Claims',
    ascending=False
).head(head)

df14 = goalies_.sort_values(
    by='Catches',
    ascending=False
).head(head)

# =========================
# CREATE SUBPLOTS
# =========================

fig = make_subplots(
    rows=4,
    cols=2,

    subplot_titles=(
        'Clean Sheets (Overall)',
        'Clean Sheets (Per Game)',

        'Saves (Overall)',
        'Saves (Per Game)',

        'High Claims (Overall)',
        'High Claims (Per Game)',

        'Catches (Overall)',
        'Catches (Per Game)'
    ),

    horizontal_spacing=0.12,
    vertical_spacing=0.08
)

# =========================
# ADD BAR CHARTS
# =========================

# Row 1
fig.add_trace(
    go.Bar(
        y=df1["Name"],
        x=df1['Clean sheets'],
        hovertext=df1['Club'],
        orientation='h',
        name='Clean Sheets Overall'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        y=df11["Name"],
        x=df11['Clean sheets'],
        hovertext=df11['Club'],
        orientation='h',
        name='Clean Sheets Per Game'
    ),
    row=1, col=2
)

# Row 2
fig.add_trace(
    go.Bar(
        y=df2["Name"],
        x=df2['Saves'],
        hovertext=df2['Club'],
        orientation='h',
        name='Saves Overall'
    ),
    row=2, col=1
)

fig.add_trace(
    go.Bar(
        y=df12["Name"],
        x=df12['Saves'],
        hovertext=df12['Club'],
        orientation='h',
        name='Saves Per Game'
    ),
    row=2, col=2
)

# Row 3
fig.add_trace(
    go.Bar(
        y=df3["Name"],
        x=df3['High Claims'],
        hovertext=df3['Club'],
        orientation='h',
        name='High Claims Overall'
    ),
    row=3, col=1
)

fig.add_trace(
    go.Bar(
        y=df13["Name"],
        x=df13['High Claims'],
        hovertext=df13['Club'],
        orientation='h',
        name='High Claims Per Game'
    ),
    row=3, col=2
)

# Row 4
fig.add_trace(
    go.Bar(
        y=df4["Name"],
        x=df4['Catches'],
        hovertext=df4['Club'],
        orientation='h',
        name='Catches Overall'
    ),
    row=4, col=1
)

fig.add_trace(
    go.Bar(
        y=df14["Name"],
        x=df14['Catches'],
        hovertext=df14['Club'],
        orientation='h',
        name='Catches Per Game'
    ),
    row=4, col=2
)

# =========================
# STYLE TRACES
# =========================

fig.update_traces(
    marker_color='rgb(110,102,250)',
    marker_line_color='rgb(8,48,107)',
    marker_line_width=2,
    opacity=0.75
)

# =========================
# UPDATE LAYOUT
# =========================

fig.update_layout(

    title={
        'text': '<b>Top Goalkeepers Statistics</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {
            'size': 28,
            'color': 'black'
        }
    },

    showlegend=False,

    autosize=False,
    width=1300,
    height=1200,

    template='ggplot2',

    paper_bgcolor='lightgray'
)

# =========================
# SHOW FIGURE
# =========================

fig.show()

### 3.2 Defenders <a class="anchor" id="3.2"></a>

In a game of footall (in fact in any game where a draw is possible) if you can not win the game then do not lose it. That means defend well and do not concede a goal. For that to happen derenders play a huge part. Good defenders are able to read the game very well and sense where the danger is in time. They know when to join the party (attacking) or sit back and defend. Although they are rare to find, there are defenders who can do more than their own job by assisting goals and scoring themselves as well. Let's see who are the best defenders at doing their job and who are contributing more than they are needed to.

**Extra info**:
Most expensive defender in the EPL:
1. Harry Maguire (Leicester City to Manchester United) - €87million
2. Virgil van Dijk (Southampton to Liverpool) - €84.65 million
3. Joao Cancelo (Juventus to Manchester City) - €65 million

Contrary to the transfer fees, big Virgil is the best defender in the list, if not in the world (of course my opinion) 

[Here is the reference.](https://www.kickoff.com/news/articles/world-news/categories/news/english-premier-league/the-10-most-expensive-defenders-of-all-time/681803?gallery=681803&gallery-page=11#ig)

In [51]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# =========================
# TOP N DEFENDERS
# =========================

top = 5

# =========================
# CREATE SUBPLOTS
# =========================

fig = make_subplots(
    rows=5,
    cols=2,

    horizontal_spacing=0.05,
    vertical_spacing=0.08,

    subplot_titles=(
        'Blocked Shots (Overall)',
        'Blocked Shots (Per Game)',

        'Interceptions (Overall)',
        'Interceptions (Per Game)',

        'Clearances (Overall)',
        'Clearances (Per Game)',

        'Headed Clearance (Overall)',
        'Headed Clearance (Per Game)',

        'Clearances Off Line (Overall)',
        'Clearances Off Line (Per Game)'
    )
)

# =========================
# OVERALL STATS
# =========================

# Blocked shots
df = defenders_38app.sort_values(
    by='Blocked shots',
    ascending=False
).head(top)

fig.add_trace(
    go.Bar(
        x=df["Name"],
        y=df['Blocked shots'],
        hovertext=df['Club']
    ),
    row=1, col=1
)

# Interceptions
df = defenders_38app.sort_values(
    by='Interceptions',
    ascending=False
).head(top)

fig.add_trace(
    go.Bar(
        x=df["Name"],
        y=df['Interceptions'],
        hovertext=df['Club']
    ),
    row=2, col=1
)

# Clearances
df = defenders_38app.sort_values(
    by='Clearances',
    ascending=False
).head(top)

fig.add_trace(
    go.Bar(
        x=df["Name"],
        y=df['Clearances'],
        hovertext=df['Club']
    ),
    row=3, col=1
)

# Headed Clearance
df = defenders_38app.sort_values(
    by='Headed Clearance',
    ascending=False
).head(top)

fig.add_trace(
    go.Bar(
        x=df["Name"],
        y=df['Headed Clearance'],
        hovertext=df['Club']
    ),
    row=4, col=1
)

# Clearances off line
df = defenders_38app.sort_values(
    by='Clearances off line',
    ascending=False
).head(top)

fig.add_trace(
    go.Bar(
        x=df["Name"],
        y=df['Clearances off line'],
        hovertext=df['Club']
    ),
    row=5, col=1
)

# =========================
# PER GAME STATS
# =========================

# Blocked shots
df = defenders_.sort_values(
    by='Blocked shots',
    ascending=False
).head(top)

fig.add_trace(
    go.Bar(
        x=df["Name"],
        y=df['Blocked shots'],
        hovertext=df['Club']
    ),
    row=1, col=2
)

# Interceptions
df = defenders_.sort_values(
    by='Interceptions',
    ascending=False
).head(top)

fig.add_trace(
    go.Bar(
        x=df["Name"],
        y=df['Interceptions'],
        hovertext=df['Club']
    ),
    row=2, col=2
)

# Clearances
df = defenders_.sort_values(
    by='Clearances',
    ascending=False
).head(top)

fig.add_trace(
    go.Bar(
        x=df["Name"],
        y=df['Clearances'],
        hovertext=df['Club']
    ),
    row=3, col=2
)

# Headed Clearance
df = defenders_.sort_values(
    by='Headed Clearance',
    ascending=False
).head(top)

fig.add_trace(
    go.Bar(
        x=df["Name"],
        y=df['Headed Clearance'],
        hovertext=df['Club']
    ),
    row=4, col=2
)

# Clearances off line
df = defenders_.sort_values(
    by='Clearances off line',
    ascending=False
).head(top)

fig.add_trace(
    go.Bar(
        x=df["Name"],
        y=df['Clearances off line'],
        hovertext=df['Club']
    ),
    row=5, col=2
)

# =========================
# STYLE ALL TRACES
# =========================

fig.update_traces(
    marker_color='rgb(96, 96, 96)',
    marker_line_color='rgb(8,48,107)',
    marker_line_width=2,
    opacity=0.75
)

# =========================
# UPDATE LAYOUT
# =========================

fig.update_layout(

    title={
        'text': '<b>Top Defender Qualities</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {
            'size': 28,
            'color': 'black'
        }
    },

    showlegend=False,

    autosize=False,

    width=1300,
    height=1300,

    template='ggplot2',

    paper_bgcolor='lightgray'
)

# =========================
# SHOW FIGURE
# =========================

fig.show()

### 3.3 Midfielders <a class="anchor" id="3.3"></a>

As it's often called mid-field is the engine-room of a football team. Mid-fielders dictate the tempo of the game, they help their attacking or defending team mates depending on the situation of their team is in. Their ability to thread in an incisive pass or their awarness to sense danger before their defending colleagues are in trouble are crucial qualities of a good mid-fielder. All great teams past and present had/have a couple world-class mid-fielders in them. Below some of these attributes are summerized.



In [52]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# =========================
# MIDFIELDER DEFENSIVE ATTRIBUTES
# =========================

mid_field_attr_D = [
    'Recoveries',
    'Duels won',
    'Successful 50/50s',
    'Aerial battles won'
]

top = 5

# =========================
# CREATE SUBPLOTS
# =========================

fig = make_subplots(
    rows=4,
    cols=2,

    subplot_titles=(
        'Recoveries (Overall)',
        'Recoveries (Per Game)',

        'Duels Won (Overall)',
        'Duels Won (Per Game)',

        'Successful 50/50s (Overall)',
        'Successful 50/50s (Per Game)',

        'Aerial Battles Won (Overall)',
        'Aerial Battles Won (Per Game)'
    ),

    horizontal_spacing=0.08,
    vertical_spacing=0.10
)

# =========================
# OVERALL STATS
# =========================

for i, atr in enumerate(mid_field_attr_D):

    df = (
        data_38app[data_38app["Position"] == 'Midfielder']
        .sort_values(by=atr, ascending=False)
        .head(top)
    )

    fig.add_trace(
        go.Bar(
            x=df['Name'],
            y=df[atr],
            hovertext=df['Club'],
            name=atr
        ),
        row=i+1,
        col=1
    )

# =========================
# PER GAME STATS
# =========================

for j, atr in enumerate(mid_field_attr_D):

    df = (
        midfielders_
        .sort_values(by=atr, ascending=False)
        .head(top)
    )

    fig.add_trace(
        go.Bar(
            x=df['Name'],
            y=df[atr],
            hovertext=df['Club'],
            name=atr
        ),
        row=j+1,
        col=2
    )

# =========================
# STYLE TRACES
# =========================

fig.update_traces(
    marker_color='rgb(110,202,82)',
    marker_line_color='rgb(8,48,107)',
    marker_line_width=2,
    opacity=0.75
)

# =========================
# UPDATE LAYOUT
# =========================

fig.update_layout(

    title={
        'text': '<b>Top Midfield Qualities: Defensive Ability</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {
            'size': 28,
            'family': 'Courier New',
            'color': 'black'
        }
    },

    showlegend=False,

    autosize=False,

    width=1200,
    height=1200,

    template='ggplot2',

    paper_bgcolor='lightgray'
)

# =========================
# SHOW FIGURE
# =========================

fig.show()

In [53]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# =========================
# MIDFIELDER CREATIVE ATTRIBUTES
# =========================

mid_field_attr_A = [
    'Assists',
    'Big chances created',
    'Cross accuracy %',
    'Through balls'
]

top = 5

# =========================
# CREATE SUBPLOTS
# =========================

fig = make_subplots(
    rows=4,
    cols=2,

    subplot_titles=(
        'Assists (Overall)',
        'Assists (Per Game)',

        'Big Chances Created (Overall)',
        'Big Chances Created (Per Game)',

        'Cross Accuracy % (Overall)',
        'Cross Accuracy % (Per Game)',

        'Through Balls (Overall)',
        'Through Balls (Per Game)'
    ),

    horizontal_spacing=0.08,
    vertical_spacing=0.10
)

# =========================
# OVERALL STATS
# =========================

for i, atr in enumerate(mid_field_attr_A):

    df = (
        data_38app[data_38app["Position"] == 'Midfielder']
        .sort_values(by=atr, ascending=False)
        .head(top)
    )

    fig.add_trace(
        go.Bar(
            x=df['Name'],
            y=df[atr],
            hovertext=df['Club'],
            name=atr
        ),
        row=i+1,
        col=1
    )

# =========================
# PER GAME STATS
# =========================

for j, atr in enumerate(mid_field_attr_A):

    df = (
        midfielders_
        .sort_values(by=atr, ascending=False)
        .head(top)
    )

    fig.add_trace(
        go.Bar(
            x=df['Name'],
            y=df[atr],
            hovertext=df['Club'],
            name=atr
        ),
        row=j+1,
        col=2
    )

# =========================
# STYLE TRACES
# =========================

fig.update_traces(
    marker_color='rgb(255, 208, 128)',
    marker_line_color='rgb(8,48,107)',
    marker_line_width=2,
    opacity=0.75
)

# =========================
# UPDATE LAYOUT
# =========================

fig.update_layout(

    title={
        'text': '<b>Top Midfield Qualities: Creative Ability</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {
            'size': 28,
            'family': 'Courier New',
            'color': 'black'
        }
    },

    showlegend=False,

    autosize=False,

    width=1200,
    height=1200,

    template='ggplot2',

    paper_bgcolor='lightgray'
)

# =========================
# SHOW FIGURE
# =========================

fig.show()

### 3.4 Forwards <a class="anchor" id="3.4"></a>

#### Goals, goals and goals:

Winning a game is the ultimate goal of the beautiful game. Scoring more goals that your opponent will do just that. We all watch football and enjoy when the team we support or our favorite team scores gaol/s. And enjoy the moment. Some would argue that goals just for the sake of goals are meaningless without attractive, entertaining attacking football. But goals goals are that wins you games. Below are the goals machines in the EPL.

1. Most goals: Sergio Aguero, Harry Kane, Jamie Vardy
2. Most right foot goals: Sergio Aguero, Harry Kane, Jamie Vardy
3. Most left foot goals: Mohammed Salah, Oliver Giroud, Ryhad Mahrez
4. Most headed goals: Oliver Giroud, Christian Benket, Andy Carrol
5. Most goal scoring nations: England, France, Brazil Argentina

In [54]:
import numpy as np
import plotly.graph_objects as go

# =========================
# TABLE COLORS
# =========================

headerColor = 'grey'
rowEvenColor = 'lightgrey'
rowOddColor = 'white'

# =========================
# TOP N PLAYERS
# =========================

head = 10

# =========================
# TABLE HEADER
# =========================

table_header = [
    'Rank',
    'Top Goal Scorers',
    'Goals with Right Foot',
    'Goals with Left Foot',
    'Headed Goals'
]

# =========================
# SORT DATA
# =========================

top_goals = (
    data_38app
    .sort_values(by='Goals', ascending=False)['Name']
    .head(head)
)

top_right_foot = (
    data_38app
    .sort_values(by='Goals with right foot', ascending=False)['Name']
    .head(head)
)

top_left_foot = (
    data_38app
    .sort_values(by='Goals with left foot', ascending=False)['Name']
    .head(head)
)

top_headed = (
    data_38app
    .sort_values(by='Headed goals', ascending=False)['Name']
    .head(head)
)

# =========================
# CREATE TABLE
# =========================

fig = go.Figure(

    data=[
        go.Table(

            header=dict(
                values=table_header,

                line_color='darkslategray',

                fill_color=headerColor,

                align='left',

                font=dict(
                    color='white',
                    size=20
                ),

                height=35
            ),

            cells=dict(

                values=[
                    list(np.arange(1, head + 1)),
                    top_goals,
                    top_right_foot,
                    top_left_foot,
                    top_headed
                ],

                fill_color=[
                    [rowOddColor, rowEvenColor] * 5
                ],

                align='left',

                height=30,

                font=dict(
                    color='black',
                    size=16,
                    family='Courier New'
                )
            )
        )
    ]
)

# =========================
# UPDATE LAYOUT
# =========================

fig.update_layout(

    title={
        'text': f'<b>TOP {head} GOAL SCORERS</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {
            'size': 26,
            'family': 'Courier New',
            'color': 'white'
        }
    },

    width=1200,
    height=550,

    template='plotly_dark'
)

# =========================
# SHOW FIGURE
# =========================

fig.show()

In [55]:
import plotly.express as px

# =========================
# GROUP GOALS BY NATIONALITY
# =========================

country_goals = (
    data.groupby('Nationality')['Goals']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

# Optional: keep only top 10 countries
country_goals = country_goals.head(10)

# =========================
# CREATE PIE CHART
# =========================

fig = px.pie(
    country_goals,

    values='Goals',
    names='Nationality',

    width=650,
    height=650,

    title='<b>Which Countries Score the Most Goals?</b>'
)

# =========================
# UPDATE PIE CHART
# =========================

fig.update_traces(
    textposition='inside',
    textinfo='percent+label'
)

# =========================
# UPDATE LAYOUT
# =========================

fig.update_layout(

    title={
        'x': 0.5,
        'xanchor': 'center',

        'font': {
            'size': 24,
            'family': 'Courier New',
            'color': 'black'
        }
    },

    showlegend=False,

    template='plotly_white'
)

# =========================
# SHOW FIGURE
# =========================

fig.show()

In [56]:
# =========================
# HEADED GOALS
# =========================

hg_f = data[data['Position'] == 'Forward']['Headed goals'].sum()

hg_m = data[data['Position'] == 'Midfielder']['Headed goals'].sum()

hg_d = data[data['Position'] == 'Defender']['Headed goals'].sum()

hg_gk = data[data['Position'] == 'Goalkeeper']['Headed goals'].sum()

# =========================
# GOALS WITH RIGHT FOOT
# =========================

rfg_f = data[data['Position'] == 'Forward']['Goals with right foot'].sum()

rfg_m = data[data['Position'] == 'Midfielder']['Goals with right foot'].sum()

rfg_d = data[data['Position'] == 'Defender']['Goals with right foot'].sum()

rfg_gk = data[data['Position'] == 'Goalkeeper']['Goals with right foot'].sum()

# =========================
# GOALS WITH LEFT FOOT
# =========================

lfg_f = data[data['Position'] == 'Forward']['Goals with left foot'].sum()

lfg_m = data[data['Position'] == 'Midfielder']['Goals with left foot'].sum()

lfg_d = data[data['Position'] == 'Defender']['Goals with left foot'].sum()

lfg_gk = data[data['Position'] == 'Goalkeeper']['Goals with left foot'].sum()

# =========================
# CREATE SUMMARY DATAFRAME
# =========================

goal_types = {
    'Position': ['Forward', 'Midfielder', 'Defender', 'Goalkeeper'],

    'Headed Goals': [
        hg_f,
        hg_m,
        hg_d,
        hg_gk
    ],

    'Right Foot Goals': [
        rfg_f,
        rfg_m,
        rfg_d,
        rfg_gk
    ],

    'Left Foot Goals': [
        lfg_f,
        lfg_m,
        lfg_d,
        lfg_gk
    ]
}

goal_summary = pd.DataFrame(goal_types)

# =========================
# DISPLAY RESULT
# =========================

print(goal_summary)

     Position  Headed Goals  Right Foot Goals  Left Foot Goals
0     Forward         388.0            1308.0            653.0
1  Midfielder         109.0             767.0            414.0
2    Defender         213.0             168.0            130.0
3  Goalkeeper           0.0               0.0              0.0


## 4. Other Statistics <a class="anchor" id="4"></a>
### 4.1 Goal distribution <a class="anchor" id="4.1"></a>
Summary:

* Forwards score more goals (obvious)
* Right-footed goals are dominant
* Defenders love to head
* The big hitters are Machester City, Liverpool and Tottenham
* The new boys (Leeds United have the fewest goals scored)
* **Crystal Palace defenders** have the best goal-scoring defenders
 

In [57]:
import plotly.graph_objects as go

# =========================
# GROUP DATA BY CLUB
# =========================

club_goals = (
    data.groupby('Club')[
        [
            'Goals with right foot',
            'Goals with left foot',
            'Headed goals'
        ]
    ]
    .sum()
    .reset_index()
)

# =========================
# CREATE FIGURE
# =========================

fig = go.Figure()

# -------------------------
# RIGHT FOOT GOALS
# -------------------------

fig.add_trace(
    go.Bar(
        x=club_goals['Club'],
        y=club_goals['Goals with right foot'],

        name='Right Foot Goals',

        marker_color='indianred'
    )
)

# -------------------------
# LEFT FOOT GOALS
# -------------------------

fig.add_trace(
    go.Bar(
        x=club_goals['Club'],
        y=club_goals['Goals with left foot'],

        name='Left Foot Goals',

        marker_color='lightsalmon'
    )
)

# -------------------------
# HEADED GOALS
# -------------------------

fig.add_trace(
    go.Bar(
        x=club_goals['Club'],
        y=club_goals['Headed goals'],

        name='Headed Goals',

        marker_color='lightseagreen'
    )
)

# =========================
# UPDATE LAYOUT
# =========================

fig.update_layout(

    barmode='group',

    title={
        'text': '<b>Goal Distribution by Club</b>',
        'x': 0.5,
        'xanchor': 'center',

        'font': {
            'size': 24,
            'family': 'Courier New',
            'color': 'black'
        }
    },

    xaxis_title='Club',
    yaxis_title='Number of Goals',

    width=1000,
    height=600,

    template='simple_white',

    xaxis_tickangle=-45
)

# =========================
# SHOW FIGURE
# =========================

fig.show()

In [58]:
import plotly.express as px

df = data

# =========================
# HEADED GOALS SUNBURST
# =========================

fig = px.sunburst(
    df,

    path=['Position', 'Club'],

    values='Headed goals',

    color='Position',

    width=650,
    height=650
)

fig.update_layout(

    title={
        'text': '<b>Headed Goals by Position and Club</b>',
        'x': 0.5,
        'xanchor': 'center',

        'font': {
            'size': 24,
            'color': 'black'
        }
    }
)

fig.show()

# =========================
# LEFT FOOT GOALS SUNBURST
# =========================

fig1 = px.sunburst(
    df,

    path=['Position', 'Club'],

    values='Goals with left foot',

    color='Position',

    width=650,
    height=650
)

fig1.update_layout(

    title={
        'text': '<b>Left Foot Goals by Position and Club</b>',
        'x': 0.5,
        'xanchor': 'center',

        'font': {
            'size': 24,
            'color': 'black'
        }
    }
)

fig1.show()

# =========================
# RIGHT FOOT GOALS SUNBURST
# =========================

fig2 = px.sunburst(
    df,

    path=['Position', 'Club'],

    values='Goals with right foot',

    color='Position',

    width=650,
    height=650
)

fig2.update_layout(

    title={
        'text': '<b>Right Foot Goals by Position and Club</b>',
        'x': 0.5,
        'xanchor': 'center',

        'font': {
            'size': 24,
            'color': 'black'
        }
    }
)

fig2.show()

In [59]:
from itertools import repeat
import plotly.graph_objects as go

# =========================
# GROUP GOALS BY CLUB
# =========================

goals_groupedby_clubs = (
    data.groupby('Club')
    .agg({
        'Goals with left foot': 'sum',
        'Goals with right foot': 'sum',
        'Headed goals': 'sum'
    })
)

# =========================
# ALL NODES
# =========================

all_nodes = [

    # Positions
    'Forwards',
    'Midfielders',
    'Defenders',

    # Goal Types
    'Right Foot Goals',
    'Left Foot Goals',
    'Headed Goals',

    # Clubs
    'Arsenal',
    'Aston Villa',
    'Brighton and Hove Albion',
    'Burnley',
    'Chelsea',
    'Crystal Palace',
    'Everton',
    'Fulham',
    'Leeds United',
    'Leicester City',
    'Liverpool',
    'Manchester City',
    'Manchester United',
    'Newcastle United',
    'Sheffield United',
    'Southampton',
    'Tottenham Hotspur',
    'West Bromwich Albion',
    'West Ham United',
    'Wolverhampton Wanderers'
]

# =========================
# SOURCE NODES
# =========================

list_1 = [0, 1, 2]
list_2 = [3, 4, 5]

source = 3 * list_1 + 20 * list_2

# =========================
# TARGET NODES
# =========================

target = []

for tar in range(3, 26):
    target.extend(repeat(tar, 3))

# =========================
# GOAL VALUES BY CLUB
# =========================

R = goals_groupedby_clubs['Goals with right foot'].values
L = goals_groupedby_clubs['Goals with left foot'].values
H = goals_groupedby_clubs['Headed goals'].values

goals_scored = []

for i in range(len(R)):
    goals_scored.append(R[i])
    goals_scored.append(L[i])
    goals_scored.append(H[i])

# =========================
# POSITIONAL GOAL VALUES
# =========================

value = [

    # Right foot goals
    rfg_f,
    rfg_m,
    rfg_d,

    # Left foot goals
    lfg_f,
    lfg_m,
    lfg_d,

    # Headed goals
    hg_f,
    hg_m,
    hg_d

] + goals_scored

# =========================
# CREATE SANKEY DIAGRAM
# =========================

fig = go.Figure(

    data=[
        go.Sankey(

            node=dict(

                pad=15,

                thickness=35,

                line=dict(
                    color="gray",
                    width=0.75
                ),

                label=all_nodes,

                color=(
                    ['#67AEE1', '#ff6e4a', '#48bf91'] +
                    3 * ['gold'] +
                    20 * ['#babad4']
                )
            ),

            link=dict(

                source=source,

                target=target,

                value=value,

                color=26 * [
                    '#94C6EA',
                    '#ffb6a4',
                    '#a3dfc8'
                ]
            )
        )
    ]
)

# =========================
# UPDATE LAYOUT
# =========================

fig.update_layout(

    title={
        'text':
        '<b>Distribution of Goals</b>'
        '<br><span style="font-size:16px;color:darkgray">'
        'By Player Position, Club, and Goal Type'
        '</span>',

        'x': 0.5,
        'xanchor': 'center',

        'font': {
            'size': 28,
            'family': 'Courier New',
            'color': 'black'
        }
    },

    width=1400,
    height=800,

    template='plotly_white'
)

# =========================
# UPDATE TEXT STYLE
# =========================

fig.update_traces(
    textfont_family='Courier New',
    selector=dict(type='sankey')
)

# =========================
# SHOW FIGURE
# =========================

fig.show()

<a href="#top">Back to top</a>  
### 4.2 Unwanted records <a class="anchor" id="4.2"></a>

Of course like in everyday-life, sometimes things happen in football wheather players like it or not. In football terms own goals, assisting the wrong player (aka error leading to a goal), being sent-off are some of the major ones. Let's see who are the unfortunate ones.

**Summary**

- Surprisingly Sergio Aguero (who's the top scorer) is also a big chance squanderer. 
- Joe Hart is error prone. No wonder Pep Guardiola has shown him the doors when he took-over at Manchester City.
- Phil Jagielka love his own net, i.e has more own goals.
- No wonder Mark Nobel is the biggest losser with so many recards to his name.
- Wondered how many times Gabriel Jesus calls his own name (swears)? 0.56 time per game (that is his big chances missed per game)


In [60]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# =========================
# UNWANTED PLAYER RECORDS
# =========================

unwanted_records = [
    'Losses',
    'Big chances missed',
    'Own goals',
    'Errors leading to goal',
    'Red cards'
]

top = 5

# =========================
# CREATE SUBPLOTS
# =========================

fig = make_subplots(
    rows=5,
    cols=2,

    subplot_titles=(
        'Losses (Overall)',
        'Losses (Per Game)',

        'Big Chances Missed (Overall)',
        'Big Chances Missed (Per Game)',

        'Own Goals (Overall)',
        'Own Goals (Per Game)',

        'Errors Leading to Goal (Overall)',
        'Errors Leading to Goal (Per Game)',

        'Red Cards (Overall)',
        'Red Cards (Per Game)'
    ),

    horizontal_spacing=0.08,
    vertical_spacing=0.10
)

# =========================
# OVERALL STATS
# =========================

for i, atr in enumerate(unwanted_records):

    df = (
        data_38app
        .sort_values(by=atr, ascending=False)
        .head(top)
    )

    fig.add_trace(
        go.Bar(
            x=df['Name'],
            y=-df[atr],

            hovertext=df['Club'],

            name=atr
        ),
        row=i+1,
        col=1
    )

# =========================
# PER GAME STATS
# =========================

for j, atr in enumerate(unwanted_records):

    df = (
        all_players
        .sort_values(by=atr, ascending=False)
        .head(top)
    )

    fig.add_trace(
        go.Bar(
            x=df['Name'],
            y=-df[atr],

            hovertext=df['Club'],

            name=atr
        ),
        row=j+1,
        col=2
    )

# =========================
# STYLE TRACES
# =========================

fig.update_traces(
    marker_color='rgb(255, 0, 0)',

    marker_line_color='darkred',

    marker_line_width=2,

    opacity=0.70
)

# =========================
# UPDATE LAYOUT
# =========================

fig.update_layout(

    title={
        'text': '<b>List of the Unfortunates</b>',
        'x': 0.5,
        'xanchor': 'center',

        'font': {
            'size': 28,
            'family': 'Courier New',
            'color': 'black'
        }
    },

    showlegend=False,

    autosize=False,

    width=1200,
    height=1100,

    template='ggplot2',

    paper_bgcolor='lightgray',

    plot_bgcolor='lightgray'
)

# =========================
# SHOW FIGURE
# =========================

fig.show()

<a href="#top">Back to top</a>